# Import libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


# Load data

In [2]:
X = pd.read_csv('../data/X.csv')
y = pd.read_csv('../data/y.csv')

# Train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,     
    random_state=42,    
    stratify=y          
)

# Train baseline

In [4]:
X_train

,X
4203,play kids paid love travel job asia 1 500 usd ...
3240,varsity technology help school nonprofit make ...
1161,10clouds innovative cutting edge fast growing ...
1282,customer service associate based san francisco...
7129,sql developer flsa exemptreports to manager jo...
...,...
6079,increasingly complex web applications mobile s...
2858,join team that s building exciting consumer ap...
477,transferwise clever new way money countries we...
5465,live push boundary interaction mobile space yo...


we make not good move so we need to invert our tokinezation

In [5]:
X_train

,X
4203,play kids paid love travel job asia 1 500 usd ...
3240,varsity technology help school nonprofit make ...
1161,10clouds innovative cutting edge fast growing ...
1282,customer service associate based san francisco...
7129,sql developer flsa exemptreports to manager jo...
...,...
6079,increasingly complex web applications mobile s...
2858,join team that s building exciting consumer ap...
477,transferwise clever new way money countries we...
5465,live push boundary interaction mobile space yo...


tf-idf requires a series so lets convert df to series

In [6]:
X_train = X_train['X']  
X_test  = X_test['X']

In [7]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = tfidf.fit_transform(X_train)
X_test  = tfidf.transform(X_test)

print("TF-IDF shape:", X_train.shape)

TF-IDF shape: (6086, 10000)


# Train model

In [8]:
lr = LogisticRegression(max_iter=1000, random_state=42)

In [9]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]  # inverse of regularization strength
}

In [10]:
X_train.shape

(6086, 10000)

In [11]:
y_train.shape

(6086, 1)

In [12]:
grid = GridSearchCV(lr, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

c:\Users\Павел\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=42),
             n_jobs=-1, param_grid={'C': [0.01, 0.1, 1, 10, 100]},
             scoring='f1_macro')

In [13]:
print("Best parameters:", grid.best_params_)
print("Best F1 score:", grid.best_score_)

Best parameters: {'C': 10}
Best F1 score: 0.7717324629644782


# Train BERT

In [14]:
# Import libraries
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torch.cuda.amp import GradScaler, autocast  # Mixed precision training

# Load data
X = pd.read_csv('../data/X.csv')['X']  # Text data
y_df = pd.read_csv('../data/y.csv')   # Label data

# Convert string labels to numerical indices
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_df.squeeze().values)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Tokenize datasets ONCE and save tensors directly to GPU
def tokenize_to_gpu(texts, device):
    encodings = tokenizer(
        texts.tolist(),
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors='pt'
    )
    return {k: v.to(device) for k, v in encodings.items()}

# Set device and use mixed precision
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move data to GPU during tokenization
train_encodings = tokenize_to_gpu(X_train, device)
test_encodings = tokenize_to_gpu(X_test, device)

# Create TensorDatasets directly on GPU
train_labels = torch.tensor(y_train, dtype=torch.long).to(device)
test_labels = torch.tensor(y_test, dtype=torch.long).to(device)

train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    train_labels
)

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask'],
    test_labels
)

# Optimized DataLoader settings
batch_size = 32  # Increased batch size for better GPU utilization
train_loader = DataLoader(
    train_dataset, 
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size,
    pin_memory=False,  # Already on GPU
    num_workers=0      # No CPU workers needed
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size * 2,  # Larger batch for evaluation
    shuffle=False,
    pin_memory=False,
    num_workers=0
)

# Initialize model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label_encoder.classes_)
).to(device)

# Optimizer with weight decay
optimizer = AdamW(model.parameters(), 
                  lr=2e-5, 
                  weight_decay=0.01)  # Regularization

# Mixed precision scaler
scaler = GradScaler()

# Training loop with optimizations
model.train()
epochs = 3

for epoch in range(epochs):
    total_loss = 0
    optimizer.zero_grad()
    
    for i, batch in enumerate(train_loader):
        b_input_ids, b_attention_mask, b_labels = batch
        
        # Mixed precision context
        with autocast():
            outputs = model(
                b_input_ids, 
                attention_mask=b_attention_mask, 
                labels=b_labels
            )
            loss = outputs.loss / 2  # Gradient accumulation scaling
        
        # Scale loss and backpropagate
        scaler.scale(loss).backward()
        
        # Update weights every 2 batches (gradient accumulation)
        if (i + 1) % 2 == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * 2  # Revert scaling for logging
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        b_input_ids, b_attention_mask, b_labels = batch
        
        outputs = model(
            b_input_ids, 
            attention_mask=b_attention_mask
        )
        
        logits = outputs.logits
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(batch_preds)
        true_labels.extend(b_labels.cpu().numpy())

print("\nOptimized BERT Performance:")
print(f"- Accuracy: {accuracy_score(true_labels, predictions):.4f}")
print(f"- F1 Score (Macro): {f1_score(true_labels, predictions, average='macro'):.4f}")

c:\Users\Павел\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Павел\AppData\Local\Temp\ipykernel_29592\3550455608.py:97: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\Павел\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
C:\Users\Павел\AppData\Local\Temp\ipykernel_29592\3550455608.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\Павел\AppData\Local\Programs\Python\Python313\Lib\site-pac

Epoch 1/3 - Loss: 1.6836
Epoch 2/3 - Loss: 1.1327
Epoch 3/3 - Loss: 0.8733

Optimized BERT Performance:
- Accuracy: 0.7070
- F1 Score (Macro): 0.7317
